# flood_solver — demostración

Solver de laberintos en **Python puro** (stdlib), método de **inundación** (flood fill = BFS).

Objetivo: resolver el laberinto **en el menor tiempo posible**, en Python puro.

Este notebook importa el solver real (`flood_solver/solver.py`) y lo compara en vivo con el solver original (`maze_api`). **El solver que se ejecuta es el `.py`** — el notebook es solo para presentar.

In [ ]:
import sys, time
from pathlib import Path

# Permite correr el notebook desde flood_solver/ o desde la raiz del repo.
REPO = Path.cwd()
if (REPO / 'flood_solver').exists():
    pass
elif (REPO.parent / 'flood_solver').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'super_solver'))

from maze_api.server import MazeGame
from maze_api.solver import solve_ascii_maze as baseline_solve
import flood_solver

try:
    from solver_v1 import solve as v1_solve
    HAVE_V1 = True
except Exception:
    HAVE_V1 = False

try:
    import super_solver as rust_mod  # solo referencia; NO elegible (final = Python)
    HAVE_RUST = True
except Exception:
    HAVE_RUST = False

print('repo:', REPO)
print('solver_v1:', HAVE_V1, ' rust(ref):', HAVE_RUST)

## 1. Correctitud — el flood produce el MISMO camino que el baseline

Validamos byte-a-byte contra el solver inicial en varios tamaños y semillas.

In [ ]:
ok = True
for n in (3, 8, 16, 32, 64):
    for seed in range(5):
        a = MazeGame(size=n, seed=seed).ascii_maze()
        base = baseline_solve(a).moves
        got = flood_solver.solve(a.encode())
        if got != base:
            ok = False
            print(f'MISMATCH N={n} seed={seed}')
print('Todas las salidas identicas al baseline:' , ok)

## 2. Comparación de tiempo (lo que decide la final)

Tiempo de cálculo del camino (best-of-N, mismo laberinto por fila).

In [ ]:
def best(fn, payload, reps):
    b = float('inf')
    for _ in range(reps):
        t0 = time.perf_counter(); fn(payload); dt = time.perf_counter() - t0
        if dt < b: b = dt
    return b * 1000.0

SIZES = [50, 100, 250, 500, 1000]   # 1000 => 1,000,000 celdas (tamano de la final)
SEED = 11

rows = []
for n in SIZES:
    a = MazeGame(size=n, seed=SEED).ascii_maze(); ab = a.encode()
    base = baseline_solve(a).moves
    flood = flood_solver.solve(ab)
    check = 'OK' if flood == base else 'FAIL'
    reps = 5 if n < 500 else 3
    tb = best(lambda s: baseline_solve(s).moves, a, 2 if n >= 500 else 3)
    tv = best(v1_solve, a, reps) if HAVE_V1 else None
    tf = best(flood_solver.solve, ab, reps + 2)
    tr = best(rust_mod.solve, a, reps + 2) if HAVE_RUST else None
    rows.append((n, tb, tv, tf, tr, check))

# Tabla
hdr = f"{'N':>5} {'celdas':>11} | {'baseline':>10} {'solver_v1':>10} {'FLOOD':>10}"
if HAVE_RUST: hdr += f" | {'rust(ref)':>10}"
hdr += f" | {'check':>6}  {'v1/flood':>9}"
print(hdr); print('-' * len(hdr))
for n, tb, tv, tf, tr, check in rows:
    line = f"{n:>5} {n*n:>11,} | {tb:>10.2f} {('-' if tv is None else f'{tv:>10.2f}')} {tf:>10.2f}"
    if HAVE_RUST: line += f" | {tr:>10.3f}"
    spd = f"{tv/tf:>8.2f}x" if tv else '   -'
    line += f" | {check:>6}  {spd:>9}"
    print(line)

## 3. Gráfico comparativo

In [ ]:
try:
    import matplotlib.pyplot as plt
    ns = [r[0] for r in rows]
    plt.figure(figsize=(8,5))
    plt.plot(ns, [r[1] for r in rows], 'o-', label='baseline (inicial)')
    if HAVE_V1: plt.plot(ns, [r[2] for r in rows], 's-', label='solver_v1 (Python)')
    plt.plot(ns, [r[3] for r in rows], '^-', label='FLOOD (nuevo)', linewidth=2)
    plt.xlabel('N (lado del laberinto)'); plt.ylabel('tiempo de calculo (ms)')
    plt.yscale('log'); plt.title('Tiempo de resolucion vs tamano'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()
except ImportError:
    print('matplotlib no disponible; la tabla de arriba es suficiente.')

## 4. Jugar el juego en vivo (opcional)

Para medir wall-clock real contra la API HTTP. Levanta el servidor en otra terminal:
```
python3 -m maze_api 200 --seed 7
```
y luego ejecuta la celda. `--play` lee el mapa una vez y reproduce el camino mínimo (1 request por movimiento, sin spam de `/state`). Si la final es **a ciegas**, usa `flood_solver.solver.play_blind(...)` en su lugar.

In [ ]:
# Descomenta para jugar contra un servidor ya levantado en 127.0.0.1:8000
# from flood_solver.solver import play_offline
# moves, timings = play_offline('127.0.0.1', 8000)
# print(timings)